# Merge NSRDB station files from Google Drive

This notebook reads the station list from CYL_geo.csv, finds matching NSRDB CSV files for each station code, skips the first two rows in each source file, uses the third row as the header, combines them, and saves one merged CSV per station in the merged_files2 folder.

In [ ]:
# Notebook description

This notebook merges NSRDB station CSVs (skipping the first two rows and using the third row as header) for station codes from CYL_geo/CyL_geo.csv, saves per-station merged files to NSRDB/merged_files2, and adds a unified datetime "timestamp" column. It then filters CYL_GHI CSVs to the 2017–2019 period, imputes missing measurements from the merged NSRDB data using a defined column mapping, and writes the results to CYL_GHI/filtered_files. A final check lists CSVs containing blank or empty values.

Inputs: CYL_geo/CyL_geo.csv, NSRDB/*.csv, CYL_GHI/*.csv
Outputs: NSRDB/merged_files2/*_merged.csv, CYL_GHI/filtered_files/*_filtered.csv

In [ ]:
import os
import glob
import pandas as pd

# Mount Google Drive when running in Colab
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
    base_dir = '/content/gdrive/MyDrive'
except Exception:
    base_dir = os.path.expanduser('~/MyDrive')

geo_path = os.path.join(base_dir, 'CYL_geo', 'CyL_geo.csv')
nsrdb_dir = os.path.join(base_dir, 'NSRDB')
merged_dir = os.path.join(nsrdb_dir, 'merged_files2')

if not os.path.exists(geo_path):
    raise FileNotFoundError(f'Geo file not found: {geo_path}')

os.makedirs(merged_dir, exist_ok=True)

geo_df = pd.read_csv(geo_path)
if 'station_code' not in geo_df.columns:
    raise KeyError('The file must contain a station_code column.')

station_codes = [str(code).strip() for code in geo_df['station_code'].dropna().unique() if str(code).strip()]

print(f'Found {len(station_codes)} station codes in {geo_path}')

for station_code in station_codes:
    pattern = os.path.join(nsrdb_dir, f'{station_code}_*.csv')
    files = sorted(glob.glob(pattern))

    if not files:
        print(f'No files found for {station_code}')
        continue

    frames = []
    for file_path in files:
        df = pd.read_csv(file_path, skiprows=2)
        frames.append(df)

    merged_df = pd.concat(frames, ignore_index=True, axis=0)

    output_path = os.path.join(merged_dir, f'{station_code}_merged.csv')
    merged_df.to_csv(output_path, index=False)

    print(f'Saved {len(files)} files for {station_code} -> {output_path}')

# Load the created merged files and build a timestamp column
merged_files = sorted(glob.glob(os.path.join(merged_dir, '*_merged.csv')))

for merged_file in merged_files:
    df = pd.read_csv(merged_file)

    required_cols = ['Year', 'Month', 'Day', 'Hour', 'Minute']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f'Skipping {merged_file}: missing columns {missing_cols}')
        continue

    df['timestamp'] = pd.to_datetime(
        df[['Year', 'Month', 'Day', 'Hour', 'Minute']].astype(str).agg(lambda r: '-'.join(r[['Year', 'Month', 'Day']]) + ' ' + ':'.join(r[['Hour', 'Minute']]), axis=1),
        format='%Y-%m-%d %H:%M',
        errors='coerce'
    )
    df = df.drop(columns=required_cols)

    df.to_csv(merged_file, index=False)
    print(f'Updated {merged_file} with timestamp column')


In [ ]:
# Filter CYL_GHI files to 2017-2019 and impute missing values from merged NSRDB data
cyl_ghi_dir = os.path.join(base_dir, 'CYL_GHI')
filtered_dir = os.path.join(cyl_ghi_dir, 'filtered_files')
os.makedirs(filtered_dir, exist_ok=True)

column_map = {
    'GHI': 'GHI',
    'wind_dir': 'Wind Direction',
    'humidity': 'Relative Humidity',
    'precipitation': 'Precipitable Water',
    'air_temp': 'Temperature',
    'wind_sp': 'Wind Speed',
    'sun_elev': 'Solar Zenith Angle',
}

for file_path in sorted(glob.glob(os.path.join(cyl_ghi_dir, '*.csv'))):
    df = pd.read_csv(file_path)

    df = df[(df['timestamp'] >= '2017-01-01') & (df['timestamp'] <= '2019-12-31 23:59:59')]

    station_code = os.path.splitext(os.path.basename(file_path))[0].split('_')[0]
    merged_station_path = os.path.join(merged_dir, f'{station_code}_merged.csv')

    if os.path.exists(merged_station_path):
        merged_df = pd.read_csv(merged_station_path, parse_dates=['timestamp'])
        merged_df['timestamp'] = pd.to_datetime(merged_df['timestamp'], errors='coerce')
        merged_df = merged_df.set_index('timestamp')
        df = df.set_index('timestamp')

        # Validate mapped columns exist in both dataframes and report missing ones
        missing_in_df = [c for c in column_map.keys() if c not in df.columns]
        missing_in_merged = [c for c in column_map.values() if c not in merged_df.columns]
        if missing_in_df:
            print(f"Warning: missing columns in {os.path.basename(file_path)}: {missing_in_df}")
        if missing_in_merged:
            print(f"Warning: missing columns in merged file {merged_station_path}: {missing_in_merged}")

        for source_col, merged_col in column_map.items():
            if source_col in df.columns and merged_col in merged_df.columns:
                fill_values = merged_df[merged_col].reindex(df.index)
                df[source_col] = df[source_col].mask(
                    df[source_col].isna() | (df[source_col].astype(str).str.strip() == ''),
                    fill_values
                )

        df = df.reset_index()
    else:
        print(f'No merged station file found for {station_code}: {merged_station_path}')

    filtered_path = os.path.join(filtered_dir, f'{station_code}_filtered.csv')
    df.to_csv(filtered_path, index=False)
    print(f'Filtered and saved {filtered_path} ({len(df)} rows)')

# inspecting excistence of blank values

In [ ]:
target_dir = os.path.join(base_dir, 'CYL_GHI')  # change this folder if needed
csv_files = sorted(glob.glob(os.path.join(target_dir, '*.csv')))

for file_path in csv_files:
    df = pd.read_csv(file_path)

    blank_mask = df.isna() | df.astype(str).apply(lambda col: col.str.strip() == '')
    blank_columns = [col for col in df.columns if blank_mask[col].any()]

    if blank_columns:
        blank_counts = {col: int(blank_mask[col].sum()) for col in blank_columns}
        print(f"{os.path.basename(file_path)}: blank values found in {blank_counts}")
    else:
        print(f"{os.path.basename(file_path)}: no blank values found")